# H11 — Relay orbit selection
Classical CR3BP dynamics and sampled geometry feed a small, independently validated relay-selection problem. QAOA runs locally. **Run All cannot submit cloud jobs or spend credits.**

The Moon is modeled as synchronously oriented with an explicitly assumed fixed rotation from terrain coordinates to synodic axes. The UTC epoch is a label for replay, not a date-specific SPICE transformation. Coverage is geometric; DSN station visibility, antenna reservations, radio-link budget, traffic capacity, station keeping and launch are outside this concept.

In [1]:
from pathlib import Path
import sys, json
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "backend/app/relay_orbits.py").exists())
sys.path[:0] = [str(root), str(root / "backend")]
from app.relay_orbits import RelayRequest, propagate_orbit, validate_selection, compare_relays, build_relay_scenario
from app.hybrid_quantum import compile_qubo, audit_mapping
from app.research_api import read_report
from quantum.relay_evidence import build_geometry
from IPython.display import display, HTML
inputs = json.loads((root / "shared/examples/relays/inputs.json").read_text())
report = read_report("relays", root / "shared/examples")
geometry = report["geometry"]
{"source": report["source_url"], "catalog_sha256": report["catalog_sha256"], "frame": report["orientation_assumption"]}

{'source': 'https://ssd-api.jpl.nasa.gov/periodic_orbits.api?sys=earth-moon&family=halo&libr=2&branch=S&periodmin=6&periodmax=9&periodunits=d',
 'catalog_sha256': 'fc170128a56df7cc144fdbb56572d3829cf0d3f8c6458b4151889f32bd423608',
 'frame': 'Ideal synchronous Moon: MOON_ME terrain directions mapped by diag(-1,-1,1) to synodic axes; no epoch-dependent libration/NAIF transform.'}

The source is NASA/JPL Three-Body Periodic Orbits API v1.0, Earth–Moon L2 southern halo family. The originally considered row 286 intersects the Moon and is retained as rejected evidence. Rows 0, 95, 190 and 240 remain candidates; no integration tolerance was loosened.

In [2]:
[(o["orbit_id"], o["period_tu"], o["jacobi"], o["stability"]) for o in inputs["orbits"]], inputs["rejected_orbits"]

([('jpl-l2-s-0', 2.0299856527687163, 3.02039371851035, 1.46270559100725),
  ('jpl-l2-s-95', 1.7906627674975273, 3.02950368041877, 1.68748222251227),
  ('jpl-l2-s-190', 1.5672976578590108, 3.04244318843518, 1.43357190445172),
  ('jpl-l2-s-240', 1.4556436750150945, 3.05087533467605, 1.19990411818932)],
 [{'orbit_id': 'jpl-l2-s-286',
   'catalog_row': 286,
   'state_nd': [1.0103798476493229,
    2.887473724726642e-27,
    -0.17250118762565506,
    -6.906147693280354e-13,
    -0.07642012279949668,
    -9.66302812886131e-12],
   'period_tu': 1.3539402995723187,
   'jacobi': 3.05996139190004,
   'stability': 1.00000000040975,
   'reason': 'Propagated trajectory intersects the lunar reference sphere',
   'sampled_minimum_radius_km': 1682.3051475658874}])

In [3]:
# Independently rerun one catalog orbit with tight integration tolerances.
trajectory = propagate_orbit(inputs["orbits"][0], inputs["system"], samples=1001, rtol=1e-12, atol=1e-14)
{k: trajectory[k] for k in ("closure_error_nd", "jacobi_drift", "minimum_moon_radius_km", "method", "rtol", "atol")}

{'closure_error_nd': 3.2924866344928424e-13,
 'jacobi_drift': 2.3296919948734285e-12,
 'minimum_moon_radius_km': 11135.988060767188,
 'method': 'DOP853',
 'rtol': 1e-12,
 'atol': 1e-14}

In [4]:
colors = ["#67e8f9", "#c4b5fd", "#fcd34d", "#fb7185"]
paths = ''.join(f'<polyline points="{a["svg_points"]}" fill="none" stroke="{colors[i]}" stroke-width=".5"/>' for i,a in enumerate(report["orbit_audits"]))
display(HTML('<p>Synodic X–Z projection, common scale. No flight or position-error claim.</p><svg viewBox="0 0 100 100" width="500" style="background:#0b1626">'+paths+'</svg>'))
[(a["orbit_id"], a["period_days"], a["closure_error_nd"], a["jacobi_drift"]) for a in report["orbit_audits"]]

[('jpl-l2-s-0',
  8.998223636699644,
  3.2924866344928424e-13,
  3.75699471533153e-12),
 ('jpl-l2-s-95',
  7.937388137634309,
  8.145063796741332e-13,
  3.481659405224491e-12),
 ('jpl-l2-s-190',
  6.947287933516118,
  2.64894376521312e-12,
  9.623857266660707e-12),
 ('jpl-l2-s-240',
  6.452364481132362,
  1.4540442567333915e-12,
  1.3602896586917268e-11)]

Terrain profiles come from saved LOLA analyses for Mons Mouton and Malapert. The terrain reference radius (1737.4 km) is explicitly reconciled with the catalog radius (1737.1 km). Both the site-to-relay ray and relay-to-Earth-center ray must clear the Moon. Equality at the horizon is blocked, following R2 semantics. Unknown terrain remains unknown.

In [5]:
geometry["direct_earth_geometry"], report["single_relay"]

({'mons-mouton-feature-center': {'state': 'available',
   'elevation_deg': 4.3714446958019915,
   'azimuth_deg': 328.88737463274043,
   'horizon_deg': 2.9203540069830276,
   'earth_center_clear': True,
   'range_km': 389566.57951024006},
  'malapert-massif-r23': {'state': 'unavailable',
   'elevation_deg': 3.7495700262261793,
   'azimuth_deg': 0.2363572451694439,
   'horizon_deg': 4.711139612883288,
   'earth_center_clear': True,
   'range_km': 389585.4557900964}},
 [{'status': 'infeasible',
   'orbit_ids': ['jpl-l2-s-0'],
   'sites': [{'site_id': 'mons-mouton-feature-center',
     'covered_s': 411900.0,
     'coverage_fraction': 0.5959201388888888,
     'maximum_gap_s': 279300.0,
     'unknown_s': 0},
    {'site_id': 'malapert-massif-r23',
     'covered_s': 616800.0,
     'coverage_fraction': 0.8923611111111112,
     'maximum_gap_s': 74400.0,
     'unknown_s': 0}],
   'issues': ['mons-mouton-feature-center: coverage below requirement',
    'mons-mouton-feature-center: communication ga

An interval counts as available only if both endpoints and its midpoint are available. The 600-second and 300-second grids are compared below. Equal aggregated classifications do not imply equal durations; coverage deltas report the difference. This does not certify every instant between samples.

In [6]:
report["convergence"]

{'changed_intervals': 0,
 'interval_count': 9216,
 'coarse_step_s': 600,
 'fine_step_s': 300,
 'coverage_deltas': [{'site_id': 'mons-mouton-feature-center',
   'orbit_id': 'jpl-l2-s-0',
   'fine_minus_coarse_covered_s': 300},
  {'site_id': 'mons-mouton-feature-center',
   'orbit_id': 'jpl-l2-s-95',
   'fine_minus_coarse_covered_s': 0},
  {'site_id': 'mons-mouton-feature-center',
   'orbit_id': 'jpl-l2-s-190',
   'fine_minus_coarse_covered_s': 0},
  {'site_id': 'mons-mouton-feature-center',
   'orbit_id': 'jpl-l2-s-240',
   'fine_minus_coarse_covered_s': 0},
  {'site_id': 'malapert-massif-r23',
   'orbit_id': 'jpl-l2-s-0',
   'fine_minus_coarse_covered_s': 0},
  {'site_id': 'malapert-massif-r23',
   'orbit_id': 'jpl-l2-s-95',
   'fine_minus_coarse_covered_s': 0},
  {'site_id': 'malapert-massif-r23',
   'orbit_id': 'jpl-l2-s-190',
   'fine_minus_coarse_covered_s': 300},
  {'site_id': 'malapert-massif-r23',
   'orbit_id': 'jpl-l2-s-240',
   'fine_minus_coarse_covered_s': 300}]}

In [7]:
# A short fresh offline check of the evidence generator.
fresh = build_geometry(inputs, duration_s=3600, step_s=600)
{"times_s": fresh["times_s"], "source_matches": fresh["source_digest"] == geometry["source_digest"], "contacts": fresh["contacts"]}

{'times_s': [0.0, 600.0, 1200.0, 1800.0, 2400.0, 3000.0, 3600.0],
 'source_matches': True,
 'contacts': {'mons-mouton-feature-center': {'jpl-l2-s-0': [True,
    True,
    True,
    True,
    True,
    True],
   'jpl-l2-s-95': [True, True, True, True, True, True],
   'jpl-l2-s-190': [True, True, True, True, True, True],
   'jpl-l2-s-240': [True, True, True, True, True, True]},
  'malapert-massif-r23': {'jpl-l2-s-0': [True, True, True, True, True, True],
   'jpl-l2-s-95': [True, True, True, True, True, True],
   'jpl-l2-s-190': [True, True, True, True, True, True],
   'jpl-l2-s-240': [True, True, True, True, True, True]}}}

Select exactly one or two distinct relay candidates. Each site has a minimum union-coverage fraction, maximum gap and total contact-duration demand. These are illustrative requirements, not NASA mission requirements. The objective minimizes declared preference costs 1,2,3,4, not physical fuel or money. For k=2, every invalid pair becomes a hard exclusion; H2 audits every binary assignment.

In [8]:
request = RelayRequest()
scenario = build_relay_scenario(geometry, request)
audit = audit_mapping(compile_qubo(scenario), scenario)
audit.model_dump()

{'binary_assignments': 256,
 'one_choice_assignments': 16,
 'feasible_assignments': 6,
 'optimal_assignments': 2,
 'ground_energy': '5',
 'ground_matches_classical': True}

In [9]:
comparison = compare_relays(geometry, request)
{"classical": comparison["classical_validation"], "quantum": comparison["quantum_validation"],
 "qubits": comparison["comparison"]["quantum"]["result"]["logical_qubits"],
 "sampled_feasible_rate": comparison["comparison"]["quantum"]["result"]["sampled_feasible_rate"],
 "sampled_optimal_rate": comparison["comparison"]["quantum"]["result"]["sampled_optimal_rate"],
 "gap": comparison["comparison"]["quantum"]["optimum_gap"], "credits_spent": comparison["comparison"]["credits_spent"]}

{'classical': {'status': 'feasible',
  'orbit_ids': ['jpl-l2-s-0', 'jpl-l2-s-240'],
  'sites': [{'site_id': 'mons-mouton-feature-center',
    'covered_s': 609600.0,
    'coverage_fraction': 0.8819444444444444,
    'maximum_gap_s': 81600.0,
    'unknown_s': 0},
   {'site_id': 'malapert-massif-r23',
    'covered_s': 691200.0,
    'coverage_fraction': 1.0,
    'maximum_gap_s': 0.0,
    'unknown_s': 0}],
  'issues': []},
 'quantum': {'status': 'feasible',
  'orbit_ids': ['jpl-l2-s-0', 'jpl-l2-s-240'],
  'sites': [{'site_id': 'mons-mouton-feature-center',
    'covered_s': 609600.0,
    'coverage_fraction': 0.8819444444444444,
    'maximum_gap_s': 81600.0,
    'unknown_s': 0},
   {'site_id': 'malapert-massif-r23',
    'covered_s': 691200.0,
    'coverage_fraction': 1.0,
    'maximum_gap_s': 0.0,
    'unknown_s': 0}],
  'issues': []},
 'qubits': 4,
 'sampled_feasible_rate': 0.8125,
 'sampled_optimal_rate': 0.21484375,
 'gap': 0.0,
 'credits_spent': 0}

In [10]:
# Stricter conditions can be infeasible; no solver silently relaxes them.
strict = compare_relays(geometry, RelayRequest(minimum_coverage=1.0, maximum_gap_s=0.0))
{"classical_status": strict["comparison"]["classical"]["result"]["status"],
 "quantum_status": strict["comparison"]["quantum"]["status"], "reason": strict["comparison"]["quantum"]["reason"]}

{'classical_status': 'infeasible',
 'quantum_status': 'not_run',
 'reason': 'classical_infeasible'}

The classical baseline enumerates at most 16 ordered assignments. Two slot encodings represent each unordered pair twice; coverage validation uses distinct orbit sets. The compact simulator uses at most four logical qubits. These small examples are classically easy, and no quantum advantage is claimed. Real hardware execution needs a separate reviewed budget and user approval.